# SCRABLE — 입력·답변 확인 → 조건 비교 → 평가자 점검

이 노트북은 **답변이 왜 그렇게 생성됐는지 확인하고 품질을 개선하기 위한 실험**입니다.
자동 프롬프트 개선은 기본으로 꺼져 있으며, 아래 세 단계의 사람 확인 기록을 통과해야 실행됩니다.

1. 리뷰 한 건의 **검색 자료·전체 system/user 입력·실제 채팅 템플릿·생성 답변**을 읽습니다.
2. 고정된 개발 리뷰 3건에서 **원래 지시문+RAG / 같은 지시문+RAG 없음 / 한 부분 수정 지시문+동일 RAG**를 비교합니다.
3. **사람이 확인한 답변 A/B**로 평가자의 점수와 근거를 점검합니다. 미확인 라벨이나 동점은 자동 통과시키지 않습니다.

**셀을 순서대로 하나씩 실행하세요.** 사람이 읽고 기록하는 셀에서는 멈춰 확인해야 합니다.
처음에는 **1~8번으로 답변 생성부터 점검**하세요. 답변이 나쁘면 실패로 기록하고 원인을 고칩니다.
9~12번은 평가자 점검, 13번은 모든 기준을 충족한 뒤 선택적으로 실행하는 자동 개선입니다.
**14번 다운로드는 어느 단계에서든 실행할 수 있습니다.** 완료한 단계마다 ZIP을 받아두면 런타임이 종료되어도 기록을 복원할 수 있습니다.

가중치를 바꿀 수 없는 것이 아닙니다. 현재 코드는 `inference_mode()`로 답변을 생성하며 **모델 가중치 업데이트는 0회**입니다.
프롬프트와 검색 근거를 바꾸는 작업도 서비스 답변 품질에 영향을 줍니다. 가중치 학습이 필요하면 별도로 SFT/LoRA 학습 과정을 추가해야 합니다.

기존 v4 코드의 실제 HyperCLOVA GPU 전체 실행은 확인했습니다. **이번 점검 흐름의 새 GPU 실행 및 품질 향상은 아직 확인하지 않았습니다.**
노트북 안의 검사 결과는 코드 검증이며, 좋은 답변·신뢰할 수 있는 평가자라는 증명이 아닙니다.

In [ ]:
#@title 1. 실행 환경 준비 — T4 GPU
import base64, hashlib, io, json, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
os.environ.update(USE_TORCH="1", USE_TF="0", USE_JAX="0")
import torch

assert torch.cuda.is_available(), "Colab 런타임을 T4 GPU로 변경한 뒤 실행하세요."
print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
PAYLOAD = "..."
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == "a2f285e9626f0e91d48e77fbc4e577f56192b1040ff3951aa06ce5d11aefa7df", "소스 파일 무결성 오류"
ROOT = Path(tempfile.mkdtemp(prefix="scrable_minimal_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(raw)) as bundle:
    for name in bundle.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("잘못된 ZIP 경로")
    bundle.extractall(ROOT)
del raw, PAYLOAD
VENV = ROOT / ".colab_venv"
INSTALL_LOG = ROOT / "install.log"
# Colab의 Python에 ensurepip가 없어도 가상환경을 만들 수 있도록 합니다.
created = subprocess.run(
    [sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(VENV)],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
INSTALL_LOG.write_text(created.stdout, encoding="utf-8")
if created.returncode:
    print(created.stdout[-8000:])
    raise RuntimeError(f"가상환경 생성 실패: {INSTALL_LOG}를 확인하세요.")
PYTHON = str(VENV / "bin" / "python")
# Colab에 설치된 pip로 지정한 가상환경에 패키지를 설치합니다.
command = [sys.executable, "-m", "pip", "--python", PYTHON,
           "install", "-q", "-e", ".[local]", "-r", "requirements-colab.txt"]
print("가상환경 생성 완료. 필요한 라이브러리를 설치합니다.")
installed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
with INSTALL_LOG.open("a", encoding="utf-8") as log:
    log.write(installed.stdout)
if installed.returncode:
    print(installed.stdout[-8000:])
    raise RuntimeError("라이브러리 설치 실패: install.log를 확인하세요.")
check = """
import torch, transformers, sentence_transformers, xgrammar
from importlib.metadata import version
assert torch.cuda.is_available(), '별도 실행 환경에서 CUDA를 사용할 수 없습니다.'
assert transformers.__version__ == '4.57.6'
assert version('huggingface-hub') == '0.36.2'
from xgrammar.contrib.hf import LogitsProcessor
info = xgrammar.TokenizerInfo([b'<eos>', b'{}', b'bad'], stop_token_ids=[0])
grammar = xgrammar.GrammarCompiler(info, max_threads=2).compile_grammar('root ::= "{}"')
processor = LogitsProcessor(grammar)
scores = processor(torch.tensor([[0]], device='cuda'), torch.zeros(1, 3, device='cuda'))
assert torch.isfinite(scores[0, 1]) and not torch.isfinite(scores[0, 0])
scores = processor(torch.tensor([[0, 1]], device='cuda'), torch.zeros(1, 3, device='cuda'))
assert torch.isfinite(scores[0, 0])
print('실행 환경 확인 완료:', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""
subprocess.check_call([PYTHON, "-c", check], cwd=ROOT)
print("코드 위치:", ROOT)
print("설치 완료. 전역 Colab 패키지는 유지했습니다. 설치 로그:", INSTALL_LOG)

In [ ]:
#@title 2. Hugging Face 토큰 인증과 모델 접근 확인
import getpass, re

# 보안 비밀을 바꾼 뒤 이 셀을 재실행하면 새 값을 사용합니다.
try:
    from google.colab import userdata
    secret = userdata.get("HF_TOKEN")
except Exception:
    secret = None
if secret and secret.strip():
    os.environ["HF_TOKEN"] = secret.strip()
del secret

preflight = r"""
import re, sys
from huggingface_hub import HfApi, get_token, hf_hub_download
from huggingface_hub.errors import GatedRepoError, LocalEntryNotFoundError
from requests.exceptions import RequestException

model = 'naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B'
token = get_token()
if not token:
    print('토큰 없음: 모델 접근이 승인된 계정의 Read 토큰이 필요합니다.', flush=True)
    sys.exit(10)

stage = 'token'
try:
    account = HfApi().whoami(token=token)
    print('토큰 인증 성공. Hugging Face 계정:', account.get('name', '(이름 없음)'), flush=True)
    stage = 'model'
    hf_hub_download(model, 'config.json', token=token)
    print('HyperCLOVA 모델 접근 확인 완료', flush=True)
except Exception as error:
    status = getattr(getattr(error, 'response', None), 'status_code', None)
    detail = str(error).replace(token, '[HF_TOKEN 숨김]')
    detail = re.sub(r'hf_[A-Za-z0-9]+', '[HF_TOKEN 숨김]', detail)
    print('실패 단계:', stage, '| 오류:', type(error).__name__, '| HTTP:', status, flush=True)
    print(detail, flush=True)
    if stage == 'token' and status == 401:
        print('토큰 인증 실패: 만료·취소되었거나 잘못된 토큰입니다.', flush=True)
        sys.exit(10)
    if isinstance(error, GatedRepoError) or status in (401, 403):
        print('접근 권한 확인: 위 계정으로 아래 모델 페이지의 이용 조건에 동의하고 접근 허용 여부를 확인하세요.', flush=True)
        print('https://huggingface.co/' + model, flush=True)
        print('접근이 허용된 같은 계정의 토큰을 사용하세요. fine-grained 토큰은 이 모델을 읽을 권한도 필요합니다.', flush=True)
    elif status == 429:
        print('Hugging Face 요청 한도에 걸렸습니다. 잠시 후 이 셀만 다시 실행하세요.', flush=True)
    elif status and status >= 500:
        print('Hugging Face 서버 응답 오류입니다. 잠시 후 이 셀만 다시 실행하세요.', flush=True)
    elif isinstance(error, (RequestException, LocalEntryNotFoundError)) and status is None:
        print('연결·시간 초과·캐시 오류입니다. 위 상세 메시지에서 연결 상태와 오프라인 설정을 확인하세요.', flush=True)
    else:
        print('위 오류 종류와 상세 메시지를 확인하세요.', flush=True)
    sys.exit(1)
"""

HF_ACCESS_LOG = ROOT / "hf_access.log"

def check_hf_access():
    result = subprocess.run(
        [PYTHON, "-u", "-c", preflight], cwd=ROOT, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    output = result.stdout
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = os.environ.get(name)
        if value:
            output = output.replace(value, "[HF_TOKEN 숨김]")
    output = re.sub(r"hf_[A-Za-z0-9]+", "[HF_TOKEN 숨김]", output)
    with HF_ACCESS_LOG.open("a", encoding="utf-8") as log:
        log.write(output)
    print(output, end="", flush=True)
    return result.returncode

HF_ACCESS_LOG.write_text("", encoding="utf-8")
access_code = check_hf_access()
if access_code == 10:
    entered = getpass.getpass("모델 접근이 승인된 Hugging Face 계정의 Read 토큰: ").strip()
    if not entered:
        raise RuntimeError("토큰이 입력되지 않았습니다. Read 토큰을 준비하고 2번 셀을 다시 실행하세요.")
    os.environ["HF_TOKEN"] = entered
    del entered
    access_code = check_hf_access()
if access_code:
    raise RuntimeError(f"모델 접근 확인 실패: 바로 위 상세 안내를 확인하세요. 진단 로그: {HF_ACCESS_LOG}")

In [ ]:
#@title 3. 한 번에 수정할 지시문 한 부분과 실행 도구
MANUAL_CHANGE = '검색 자료에 등장하는 다른 고객의 신분·증상·경험을 현재 고객의 사실로 옮기지 마세요. 현재 고객의 상황은 review에 명시된 내용으로만 한정하세요.' #@param {type:"string"}
WORK_FOLDER = "outputs/quality_workflow" #@param {type:"string"}

from IPython.display import HTML, display
from google.colab import files
import re

SPEC_PATH = ROOT / "configs/quality_workflow.json"
spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
WF_DIR = (ROOT / WORK_FOLDER).resolve()
if not WF_DIR.is_relative_to(ROOT.resolve()):
    raise ValueError("작업 폴더는 프로젝트 폴더 안에 지정하세요.")
spec["experiment"]["output_dir"] = str(WF_DIR)
SPEC_PATH.write_text(json.dumps(spec, ensure_ascii=False, indent=2), encoding="utf-8")
(ROOT / spec["manual_addendum_file"]).write_text(MANUAL_CHANGE.strip() + "\n", encoding="utf-8")

RUN_ENV = os.environ.copy()
RUN_ENV.update(USE_TORCH="1", USE_TF="0", USE_JAX="0", TOKENIZERS_PARALLELISM="false")

def run_quality(action, answers=None, enable=False, backup=None):
    global process
    previous = globals().get("process")
    if isinstance(previous, subprocess.Popen) and previous.poll() is None:
        raise RuntimeError("이전 작업이 실행 중입니다. 중복 실행하지 말고 완료를 기다리세요.")
    command = [PYTHON, "-u", "-m", "scrable.quality_workflow", action, "--spec", str(SPEC_PATH)]
    if answers is not None:
        answer_path = ROOT / "human_review_input.json"
        answer_path.write_text(json.dumps(answers, ensure_ascii=False, indent=2), encoding="utf-8")
        command += ["--answers", str(answer_path)]
    if enable:
        command.append("--enable")
    if backup is not None:
        command += ["--backup", str(backup)]
    with (ROOT / "quality_commands.log").open("a", encoding="utf-8") as log:
        log.write("\nACTION: " + action + "\n")
        process = subprocess.Popen(command, cwd=ROOT, env=RUN_ENV, text=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        try:
            for line in process.stdout:
                for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
                    secret = os.environ.get(key)
                    if secret:
                        line = line.replace(secret, "[HF_TOKEN 숨김]")
                line = re.sub(r"hf_[A-Za-z0-9]{10,}", "[HF_TOKEN 숨김]", line)
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            process.stdout.close()
    if status:
        raise RuntimeError("이 단계는 완료되지 않았습니다. 바로 위 안내를 확인하세요. 14번으로 현재 기록을 다운로드할 수 있습니다.")

def current_state():
    output = subprocess.check_output(
        [PYTHON, "-m", "scrable.quality_workflow", "status", "--spec", str(SPEC_PATH)],
        cwd=ROOT, env=RUN_ENV, text=True,
    )
    return json.loads(output)

def current_trial():
    return Path(current_state()["trial_dir"])

def show_report(name):
    path = current_trial() / name
    if not path.exists():
        raise RuntimeError("앞 단계를 실행해 보고서를 먼저 만드세요.")
    display(HTML(path.read_text(encoding="utf-8")))

print("개발 리뷰 3건, validation 3건, 최종 test 5건을 분리합니다.")
print("답변 비교에서는 개발 리뷰만 사용합니다. 자동 개선은 기본 OFF입니다.")
print("변경할 한 부분:", MANUAL_CHANGE)

In [ ]:
#@title 4. 데이터 준비 — 이전 품질 점검 ZIP 복원은 선택
RESTORE_CHECKPOINT = False #@param {type:"boolean"}
if RESTORE_CHECKPOINT:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("scrable_quality_results.zip 한 개를 선택하세요.")
    name, content = next(iter(uploaded.items()))
    backup_path = ROOT / "restore_quality.zip"
    backup_path.write_bytes(content)
    run_quality("restore", backup=backup_path)
    print("백업에 저장된 지시문과 검토 기록을 복원했습니다.")
run_quality("prepare")
print("현재 점검 기록:", current_trial())

In [ ]:
#@title 5. 단계 1 — 리뷰 한 건의 전체 입력과 답변 확인
run_quality("trace")
show_report("stage1_trace.html")

In [ ]:
#@title 6. 단계 1 기록 — 결과가 나빠도 관찰한 사실을 적으세요
READ_FULL_INPUT_AND_OUTPUT = False #@param {type:"boolean"}
OBSERVATION = "" #@param {type:"string"}
# 무엇을 넣었고 무엇이 나왔는지 확인했다는 기록입니다. 좋은 답변이라는 선언이 아닙니다.
if READ_FULL_INPUT_AND_OUTPUT:
    run_quality("review-trace", {"reviewed": True, "note": OBSERVATION})
else:
    print("5번의 원문·검색 자료·전체 입력·답변을 읽고 체크 및 관찰 메모를 입력하세요.")

In [ ]:
#@title 7. 단계 2 — RAG 유무와 지시문 한 부분 변경을 따로 비교
run_quality("compare")
show_report("stage2_comparison.html")

In [ ]:
#@title 8. 단계 2 기록 — 선택한 조건의 답변을 리뷰별로 확인
READ_ALL_COMPARISONS = False #@param {type:"boolean"}
SELECTED_CONDITION = "미선택" #@param ["미선택", "baseline_rag_on", "baseline_rag_off", "manual_rag_on"]
# RAG_EFFECT: helps=도움, hurts=방해, unclear=뚜렷한 차이 없음
# PROMPT_EFFECT: baseline=기존이 나음, manual=수정이 나음, tie=비슷함, both_bad=둘 다 문제
# 아래 네 항목은 선택 조건의 답변에 대해 각각 판정합니다. 모르면 미확인으로 남기세요.
# 문제가 있는 항목만 실패로 기록하세요. 실패 기록도 저장되며 자동 개선은 열리지 않습니다.
RAG_EFFECT_1 = "미선택" #@param ["미선택", "helps", "hurts", "unclear"]
PROMPT_EFFECT_1 = "미선택" #@param ["미선택", "baseline", "manual", "tie", "both_bad"]
LANGUAGE_1 = "미확인" #@param ["미확인", "통과", "실패"]
CURRENT_FACTS_1 = "미확인" #@param ["미확인", "통과", "실패"]
SUPPORTED_CLAIMS_1 = "미확인" #@param ["미확인", "통과", "실패"]
REPLY_FORMAT_1 = "미확인" #@param ["미확인", "통과", "실패"]
REVIEW_NOTE_1 = "" #@param {type:"string"}
RAG_EFFECT_2 = "미선택" #@param ["미선택", "helps", "hurts", "unclear"]
PROMPT_EFFECT_2 = "미선택" #@param ["미선택", "baseline", "manual", "tie", "both_bad"]
LANGUAGE_2 = "미확인" #@param ["미확인", "통과", "실패"]
CURRENT_FACTS_2 = "미확인" #@param ["미확인", "통과", "실패"]
SUPPORTED_CLAIMS_2 = "미확인" #@param ["미확인", "통과", "실패"]
REPLY_FORMAT_2 = "미확인" #@param ["미확인", "통과", "실패"]
REVIEW_NOTE_2 = "" #@param {type:"string"}
RAG_EFFECT_3 = "미선택" #@param ["미선택", "helps", "hurts", "unclear"]
PROMPT_EFFECT_3 = "미선택" #@param ["미선택", "baseline", "manual", "tie", "both_bad"]
LANGUAGE_3 = "미확인" #@param ["미확인", "통과", "실패"]
CURRENT_FACTS_3 = "미확인" #@param ["미확인", "통과", "실패"]
SUPPORTED_CLAIMS_3 = "미확인" #@param ["미확인", "통과", "실패"]
REPLY_FORMAT_3 = "미확인" #@param ["미확인", "통과", "실패"]
REVIEW_NOTE_3 = "" #@param {type:"string"}

if READ_ALL_COMPARISONS:
    run_quality("comparison-template")
    answers = json.loads((current_trial() / "comparison-template.json").read_text(encoding="utf-8"))
    answers["selected_condition"] = SELECTED_CONDITION
    for index, entry in enumerate(answers["entries"].values(), 1):
        entry.update(rag_effect=globals()[f"RAG_EFFECT_{index}"],
                     prompt_effect=globals()[f"PROMPT_EFFECT_{index}"],
                     note=globals()[f"REVIEW_NOTE_{index}"])
        for key, field in (("language_ok", "LANGUAGE"), ("current_customer_facts_only", "CURRENT_FACTS"),
                           ("supported_claims_only", "SUPPORTED_CLAIMS"), ("reply_format_ok", "REPLY_FORMAT")):
            entry[key] = {"통과": True, "실패": False}.get(globals()[f"{field}_{index}"])
    run_quality("review-comparison", answers)
else:
    print("7번의 세 조건을 리뷰별로 읽고 선택과 관찰 결과를 기록하세요.")

In [ ]:
#@title 9. 단계 3 준비 — 사람이 확인할 답변 A/B 초안
run_quality("pair-template")
pair_data = json.loads((current_trial() / "human_pairs.json").read_text(encoding="utf-8"))
for pair in pair_data["pairs"]:
    print("\n", pair["pair_id"], " / ", pair["diagnostic_focus"])
    print("리뷰:", pair["review"]["review"])
    print("A:", pair["answer_a"])
    print("B:", pair["answer_b"])
    print("사람이 원문과 두 답변을 확인하기 전에는 정답 라벨이 없습니다.")

In [ ]:
#@title 10. 단계 3 사람 판정 — A/B를 읽고 필요하면 초안도 수정
# 이 셀은 사람의 선택을 기록합니다. 미선택 상태를 AI가 대신 채우지 않습니다.
PAIR_1_A = 'Sorry the delivery address was incorrect. Could you clarify whether the corrected address is now saved and what issue remains?' #@param {type:"string"}
PAIR_1_B = '배송 주소에 문제가 있었군요. 수정한 주소가 현재 저장되어 있는지와 남은 문제를 알려주시겠어요?' #@param {type:"string"}
PAIR_1_BETTER = "미선택" #@param ["미선택", "A", "B"]
PAIR_1_REVIEWED = False #@param {type:"boolean"}
PAIR_1_REASON = "" #@param {type:"string"}
PAIR_2_A = 'We are sorry you have not received a response. Could you share the complaint details and when you contacted support?' #@param {type:"string"}
PAIR_2_B = 'We have already refunded all of your orders and guarantee a reply within five minutes.' #@param {type:"string"}
PAIR_2_BETTER = "미선택" #@param ["미선택", "A", "B"]
PAIR_2_REVIEWED = False #@param {type:"boolean"}
PAIR_2_REASON = "" #@param {type:"string"}
PAIR_3_A = 'Sorry the app has been unreliable. Which feature fails, and what error do you see?' #@param {type:"string"}
PAIR_3_B = 'As a student whose payment was withdrawn before an order was cancelled, you should receive a refund in three days.' #@param {type:"string"}
PAIR_3_BETTER = "미선택" #@param ["미선택", "A", "B"]
PAIR_3_REVIEWED = False #@param {type:"boolean"}
PAIR_3_REASON = "" #@param {type:"string"}

pair_data = json.loads((current_trial() / "human_pairs.json").read_text(encoding="utf-8"))
for index, pair in enumerate(pair_data["pairs"], 1):
    pair.update(answer_a=globals()[f"PAIR_{index}_A"], answer_b=globals()[f"PAIR_{index}_B"],
                expected_better=globals()[f"PAIR_{index}_BETTER"],
                reviewed_by_human=globals()[f"PAIR_{index}_REVIEWED"],
                note=globals()[f"PAIR_{index}_REASON"])
run_quality("save-pairs", pair_data)

In [ ]:
#@title 11. 단계 3 — 사람의 판정을 숨기고 평가자의 점수·근거 확인
run_quality("calibrate")
show_report("stage3_calibration.html")
print("점수가 같거나 사람의 판정과 다르면 자동 개선은 통과하지 않습니다.")

In [ ]:
#@title 12. 평가 근거 확인 — 점수만 맞고 이유는 틀리지 않았는지 읽으세요
READ_JUDGE_RATIONALES = False #@param {type:"boolean"}
RATIONALES_FIT_CRITERIA = "미확인" #@param ["미확인", "통과", "실패"]
JUDGE_REVIEW_NOTE = "" #@param {type:"string"}
if READ_JUDGE_RATIONALES:
    run_quality("review-judge", {"rationales_checked": True,
                                "rationales_acceptable": {"통과": True, "실패": False}.get(RATIONALES_FIT_CRITERIA),
                                "note": JUDGE_REVIEW_NOTE})
else:
    print("11번에서 항목별 점수와 판단 근거를 확인하고 기록하세요.")

In [ ]:
#@title 13. 자동 프롬프트 개선 — 선택 사항, 기본 OFF
RUN_AUTOMATIC_OPTIMIZATION = False #@param {type:"boolean"}
if RUN_AUTOMATIC_OPTIMIZATION:
    run_quality("automatic", enable=True)
    print("결과는 새 후보입니다. 점수 상승만으로 서비스용으로 확정하지 않습니다.")
else:
    print("자동 개선 OFF. 먼저 입력·답변 확인, 조건 비교, 평가자 점검을 진행하세요.")

In [ ]:
#@title 14. 현재까지의 모든 점검 기록 다운로드 — 어느 단계에서든 실행 가능
run_quality("export")
result_zip = ROOT / "scrable_quality_results.zip"
# 환경 로그에는 토큰 값을 기록하지 않습니다.
with zipfile.ZipFile(result_zip, "a", zipfile.ZIP_DEFLATED) as bundle:
    for log_name in ("quality_commands.log", "install.log", "hf_access.log"):
        log_path = ROOT / log_name
        if log_path.exists():
            bundle.write(log_path, log_name)
files.download(str(result_zip))
print("검토 기록을 다운로드했습니다. 새 런타임에서는 4번의 복원 기능으로 이어갈 수 있습니다.")